# Features_selection Nickels_Ethan

> **Objectif :** Réduire la dimensionnalité des 103 variables issues du prétraitement (Phase 2) en
> comparant trois approches de sélection de variables, en vue de la modélisation (Phase 4).
>
> **Auteurs :** Nickels Ethan

---
## Table des matières
1. [Importation des librairies et des données](#section-1)
2. [Sélection personnelle (P)](#section-2)
3. [Sélection par Variance Threshold (VT)](#section-3)
4. [Sélection par Select K-Best (KB)](#section-4)
5. [Comparaison des sélections](#section-5)
6. [Sauvegarde des résultats](#section-6)

---
## 1. Importation des librairies et des données <a id='section-1'></a>
On repart des 4 fichiers CSV produits en Phase 2 (`X_train`, `X_test`, `y_train`, `y_test`), déjà
prétraités (encodés et standardisés).

In [7]:
import pandas as pd
import numpy as np

from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif

X_train = pd.read_csv('X_train_preprocessed.csv')
X_test = pd.read_csv('X_test_preprocessed.csv')
y_train = pd.read_csv('y_train_preprocessed.csv').squeeze()
y_test = pd.read_csv('y_test_preprocessed.csv').squeeze()

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)

X_train : (32561, 104)
X_test  : (16281, 104)


---
## 2. Sélection personnelle (P) <a id='section-2'></a>
Sur base de l'analyse exploratoire (Phase 1) et de la matrice de corrélation, on sélectionne
manuellement les variables jugées les plus pertinentes métier : les 5 variables numériques
(fortement liées au revenu), le sexe, le statut marital, la relation et un sous-ensemble ciblé de
niveaux d'éducation supérieure (les plus discriminants observés en Phase 1).

In [8]:
def select_personal(X, base_numeric, keywords):
    """Sélectionne les colonnes numériques de base plus toute colonne encodée contenant
    l'un des mots-clés fournis (ex. 'sex_', 'marital_status_')."""
    cols = list(base_numeric)
    for kw in keywords:
        cols += [c for c in X.columns if c.startswith(kw)]
    return [c for c in cols if c in X.columns]

base_numeric = ['num__age', 'num__education_num', 'num__capital_gain', 'num__capital_loss',
                'num__hours_per_week']
keywords_P = ['cat__sex_', 'cat__marital_status_', 'cat__relationship_']

selected_P = select_personal(X_train, base_numeric, keywords_P)
print(f"Sélection personnelle : {len(selected_P)} variables")
print(selected_P)

X_train_P = X_train[selected_P]
X_test_P = X_test[selected_P]

Sélection personnelle : 20 variables
['num__age', 'num__education_num', 'num__capital_gain', 'num__capital_loss', 'num__hours_per_week', 'cat__sex_Female', 'cat__sex_Male', 'cat__marital_status_Divorced', 'cat__marital_status_Married-AF-spouse', 'cat__marital_status_Married-civ-spouse', 'cat__marital_status_Married-spouse-absent', 'cat__marital_status_Never-married', 'cat__marital_status_Separated', 'cat__marital_status_Widowed', 'cat__relationship_Husband', 'cat__relationship_Not-in-family', 'cat__relationship_Other-relative', 'cat__relationship_Own-child', 'cat__relationship_Unmarried', 'cat__relationship_Wife']


La sélection personnelle retient les variables numériques les plus corrélées au revenu (Phase 1) et
trois blocs catégoriels jugés déterminants par l'analyse métier : `sex`, `marital_status` et
`relationship`, qui montraient tous un lien visuellement marqué avec le revenu en section 9 de la
Phase 1. Ce choix est délibérément resserré (peu de variables) pour rester interprétable.

---
## 3. Sélection via Variance Threshold (VT) <a id='section-3'></a>
`VarianceThreshold` élimine les variables dont la variance est trop faible pour être informative
(ex. colonnes One-Hot quasi constantes, une modalité rare présente sur presque aucun individu). On
fixe un seuil de 0.01, adapté à des variables binaires (une variance de 0.01 correspond à une
modalité présente chez environ 1% ou 99% des individus).

In [9]:
def select_variance_threshold(X_train, X_test, threshold=0.01):
    """Sélectionne les variables dont la variance dépasse un seuil donné."""
    selector = VarianceThreshold(threshold=threshold)
    selector.fit(X_train)
    selected_cols = X_train.columns[selector.get_support()]
    return X_train[selected_cols], X_test[selected_cols], list(selected_cols)

X_train_VT, X_test_VT, selected_VT = select_variance_threshold(X_train, X_test, threshold=0.01)
print(f"Sélection Variance Threshold : {len(selected_VT)} variables (sur {X_train.shape[1]})")

Sélection Variance Threshold : 56 variables (sur 104)


La Variance Threshold élimine principalement des modalités très rares issues du One-Hot Encoding
(ex. `native_country_Holand-Netherlands`, `workclass_Without-pay`), qui apportent peu de signal
statistique car présentes chez une poignée d'individus seulement. Contrairement à la sélection
personnelle, cette méthode ne tient pas compte de la variable cible : elle peut donc conserver des
variables peu prédictives (mais variées) et en éliminer d'autres qui, bien que rares, seraient
discriminantes.

---
## 4. Sélection via Select K-Best (KB) <a id='section-4'></a>
`SelectKBest` sélectionne les K variables les plus corrélées à la cible selon un test statistique
(ici ANOVA F-test, adapté à des features numériques/binaires et une cible catégorielle). On choisit
K=20, un compromis entre réduction de dimension et conservation d'information.

In [10]:
def select_k_best(X_train, X_test, y_train, k=20):
    """Sélectionne les k variables les plus corrélées à la cible via un test ANOVA F."""
    selector = SelectKBest(score_func=f_classif, k=k)
    selector.fit(X_train, y_train)
    selected_cols = X_train.columns[selector.get_support()]
    scores = pd.Series(selector.scores_, index=X_train.columns).sort_values(ascending=False)
    return X_train[selected_cols], X_test[selected_cols], list(selected_cols), scores

X_train_KB, X_test_KB, selected_KB, kb_scores = select_k_best(X_train, X_test, y_train, k=20)
print(f"Sélection Select K-Best : {len(selected_KB)} variables")
kb_scores.head(20)

Sélection Select K-Best : 20 variables


cat__marital_status_Married-civ-spouse    8025.842062
cat__relationship_Husband                 6240.018276
num__education_num                        4120.095780
cat__marital_status_Never-married         3674.200147
num__age                                  1886.707314
num__hours_per_week                       1813.386282
cat__relationship_Own-child               1794.157489
num__capital_gain                         1709.150064
cat__sex_Female                           1593.107907
cat__sex_Male                             1593.107907
cat__occupation_Exec-managerial           1575.848402
cat__relationship_Not-in-family           1199.481344
cat__education_Bachelors                  1096.321775
cat__education_Masters                    1023.409969
cat__occupation_Other-service              815.835009
cat__education_Prof-school                 800.498796
num__capital_loss                          754.830452
cat__relationship_Unmarried                678.313649
cat__workclass_Self-emp-inc 

Les variables les mieux classées par le test ANOVA F sont `marital_status_Married-civ-spouse`,
`relationship_Husband`/`Wife`, `education_num`, `age`, `capital_gain` et `hours_per_week` — ce qui
confirme statistiquement les tendances déjà observées visuellement en Phase 1. Cette convergence
entre l'intuition métier (sélection P) et le test statistique (KB) est un bon signe de cohérence des
analyses.

---
## 5. Comparaison des sélections <a id='section-5'></a>
On compare le nombre de variables retenues par chaque méthode, ainsi que leur recouvrement.

In [11]:
comparison = pd.DataFrame({
    'Méthode': ['Personnelle (P)', 'Variance Threshold (VT)', 'Select K-Best (KB)'],
    'Nb variables': [len(selected_P), len(selected_VT), len(selected_KB)]
})
print(comparison)

overlap_P_KB = set(selected_P) & set(selected_KB)
print(f"\nRecouvrement P / KB : {len(overlap_P_KB)} variables communes -> {sorted(overlap_P_KB)}")

                   Méthode  Nb variables
0          Personnelle (P)            20
1  Variance Threshold (VT)            56
2       Select K-Best (KB)            20

Recouvrement P / KB : 13 variables communes -> ['cat__marital_status_Married-civ-spouse', 'cat__marital_status_Never-married', 'cat__relationship_Husband', 'cat__relationship_Not-in-family', 'cat__relationship_Own-child', 'cat__relationship_Unmarried', 'cat__sex_Female', 'cat__sex_Male', 'num__age', 'num__capital_gain', 'num__capital_loss', 'num__education_num', 'num__hours_per_week']


La sélection VT est de loin la moins restrictive (elle ne fait qu'éliminer les variables quasi
constantes), tandis que P et KB réduisent beaucoup plus fortement la dimension. Le recouvrement
important entre P et KB confirme que l'intuition métier (Phase 1) était globalement alignée avec le
test statistique, ce qui renforce la confiance dans ces deux sélections pour la Phase 4.

---
## 6. Sauvegarde des résultats <a id='section-6'></a>
On sauvegarde les 8 fichiers CSV attendus par les consignes : les jeux `X` (train/test) pour chacune
des trois sélections, ainsi que la cible `y` (train/test), commune aux trois sélections puisqu'elle
n'est pas affectée par la réduction de variables.

In [12]:
X_train_P.to_csv('xtrain_P.csv', index=False)
X_test_P.to_csv('xtest_P.csv', index=False)
X_train_VT.to_csv('xtrain_VT.csv', index=False)
X_test_VT.to_csv('xtest_VT.csv', index=False)
X_train_KB.to_csv('xtrain_KB.csv', index=False)
X_test_KB.to_csv('xtest_KB.csv', index=False)
y_train.to_csv('ytrain.csv', index=False)
y_test.to_csv('ytest.csv', index=False)

print("8 fichiers CSV sauvegardés : xtrain_P, xtest_P, xtrain_VT, xtest_VT, "
      "xtrain_KB, xtest_KB, ytrain, ytest (.csv)")

8 fichiers CSV sauvegardés : xtrain_P, xtest_P, xtrain_VT, xtest_VT, xtrain_KB, xtest_KB, ytrain, ytest (.csv)


Les 8 fichiers CSV sont sauvegardés dans le répertoire courant. Ils seront réutilisés indépendamment
par les 6 notebooks de modélisation de la Phase 4, chacun combinant une sélection de variables (P,
VT ou KB) avec un algorithme (KNN, Decision Tree ou Random Forest).